In [ ]:
# Setup: run first. All default exercises use only the Python standard library.
import math
import random
from contextlib import redirect_stdout
from io import StringIO

random.seed(71)

def check_test(name, actual, expected, tol=1e-3):
    """Print a visible result, including unfinished or malformed answers."""
    def same(a, b):
        if isinstance(b, bool):
            return isinstance(a, bool) and a == b
        if isinstance(b, (int, float)):
            return isinstance(a, (int, float)) and not isinstance(a, bool) and math.isclose(a, b, abs_tol=tol, rel_tol=0)
        if isinstance(b, (list, tuple)):
            return isinstance(a, type(b)) and len(a) == len(b) and all(same(x, y) for x, y in zip(a, b))
        if isinstance(b, dict):
            return isinstance(a, dict) and a.keys() == b.keys() and all(same(a[k], b[k]) for k in b)
        return a == b
    try:
        matched = same(actual, expected)
        print(f"✅ {name}: {actual!r}" if matched else f"❗ {name}: {actual!r} (Expected: {expected!r})")
        return matched
    except Exception as exc:
        print(f"❗ {name}: {actual!r} (Expected: {expected!r}; {type(exc).__name__}: {exc})")
        return False

def run_check(name, calculation, expected, tol=1e-3):
    """Evaluate inside the exception boundary, including errors in learner code."""
    try:
        return check_test(name, calculation(), expected, tol)
    except Exception as exc:
        print(f"❗ {name}: {type(exc).__name__}: {exc} (Expected: {expected!r})")
        return False

# Opt in separately after reading the resource note below. No automatic installs.
RUN_XNLI = False
RUN_EMBEDDINGS = False
RUN_FINETUNING = False
tok = model = embedding_model = None

## Intuition & Quick Walkthrough

**Multilingual NLP — Build from Scratch Practice**

Source: [Build It, Steps 1–3](../docs/en.md#build-it), with
[code/main.py](../code/main.py) supplying the Extra exercises.
Fill each `pass`, then rerun its cell. A red `❗` is expected while an exercise
is unfinished; `✅` means that particular check matched, not that the model is accurate.

The same idea travels through three representations: NLI logits become label
scores, sentence vectors become similarity scores, and labeled examples become
padded training batches. First implement their small numerical/data operations
with Python loops and `math`; then connect them to the lesson's real APIs.
This notebook does not ask you to rebuild or pretrain XLM-R from scratch.

The sandbox uses **invented four-way logits and four-dimensional geometry**,
not a language model, translation lookup, or fake API. Its data differs from
the lesson and all exercise checks. Change the values and observe the outputs.
For logits $z$, softmax uses $e^{z_i-m}/\sum_j e^{z_j-m}$ with $m=\max_j z_j$;
vector alignment uses $(u\cdot v)/(\|u\|\|v\|)$.

In [ ]:
# Runnable sandbox only. These are numerical examples, not model predictions.
preview_logits = [-1.7, 0.4, 2.3, -0.2]
preview_shift = max(preview_logits)
preview_weights = [math.exp(value - preview_shift) for value in preview_logits]
preview_probs = [weight / sum(preview_weights) for weight in preview_weights]
print("Four-way probabilities:", [round(value, 4) for value in preview_probs])
print("Probability total:", sum(preview_probs))

preview_u, preview_v = [2, -1, 4, 6], [1, 3, 5, 2]
preview_dot = sum(left * right for left, right in zip(preview_u, preview_v))
preview_norms = math.sqrt(sum(x*x for x in preview_u)) * math.sqrt(sum(x*x for x in preview_v))
print("Vector dimensions:", len(preview_u), len(preview_v))
print("Directional similarity:", round(preview_dot / preview_norms, 4))

preview_sequences = [[81, 84, 87, 89], [93], [95, 97]]
preview_width = max(len(row) for row in preview_sequences)
preview_padded = [row + [99] * (preview_width - len(row)) for row in preview_sequences]
print("Unequal sequence lengths:", [len(row) for row in preview_sequences])
print("Rectangular batch (99 is the example padding ID):", preview_padded)

### Scope and optional real-model runs

The three numbered Steps match the docs one-to-one. Functions marked **practice
helper** are new exercises exposing the operations inside those steps; original
functions retain their exact signatures, including the nested `tokenize_fn(ex)`.
The reference has no type annotations, so none are invented for its functions.
Extras preserve every function from `main.py`, including its reporting `main()`.

Default mode **stays stdlib-first for educational clarity**. Production imports
are optional to reproduce the real calls requested in `note.md`; they are never
replaced by recording classes or fabricated predictions. In a separate learning
environment those cells need `torch`, `transformers`, `sentence-transformers`,
`numpy`, `datasets`, `accelerate`, and tokenizer support such as `sentencepiece`.
This notebook installs nothing and changes no repository dependencies.

**Before opting in:** do you consent to downloading/loading the selected model
and its memory use? Leave all flags `False` for the lightweight route. Set only
the desired flag to `True` and run its cell after reviewing this table.

| Flag | Download and storage | Estimated memory / alternative |
|---|---|---|
| `RUN_XNLI` | XLM-R XNLI weights: about **2.24 GB**, plus tokenizer/config | Budget roughly **4–8 GB RAM** for short CPU inference; implementation uses CPU, no GPU required. Use numerical exercises for zero downloads. |
| `RUN_EMBEDDINGS` | Multilingual MiniLM weights: about **471 MB**, plus tokenizer/config | Roughly **1–2 GB RAM** for short CPU inference; smaller than XNLI, but serves embeddings rather than NLI. |
| `RUN_FINETUNING` | Uses your explicitly prepared task model; no loader in this cell | Full FP32 Adam training of the large XNLI model can require **9 GB or more just for weights, gradients and optimizer states**, plus activations. Use a suitably provisioned cloud notebook or stay with the batch exercises. |

Default Hugging Face model storage is `~/.cache/huggingface/hub/` unless your
environment overrides it. Training writes under `out/` relative to the kernel
working directory. Memory estimates are planning estimates, not measurements;
actual use depends on batch size, sequence length, dtype and software version.
Keep one model loaded at a time when memory is limited.

Weight sizes checked on 2026-10-09: [XNLI files](https://huggingface.co/joeddav/xlm-roberta-large-xnli/tree/main),
[MiniLM weights](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/blob/main/model.safetensors).
No model weights were downloaded to create or verify this scaffold.

## Step 1: zero-shot cross-lingual classification

NLI asks whether a premise supports a hypothesis. For each candidate label,
the docs build a hypothesis and keep its **entailment** probability.

Implement these **practice helpers** without NumPy or Torch:

- `stable_softmax(logits)` — $p_i=\exp(z_i-m)/\sum_j\exp(z_j-m)$,
  where $m=\max_j z_j$. Return a list; return `[]` for empty input.
- `rank_entailment(label_logits, entailment_index=2)` — for each label $\ell$,
  $s_\ell=\mathrm{softmax}(z_\ell)_e$. Return a dict in descending score order;
  preserve input order for ties. The mapping values are equally sized, nonempty
  logit lists and the index is valid. Empty mapping gives `{}`.

These are explicit numerical fixtures, **not predictions**. Probabilities sum
to one within each NLI triple, not across candidate labels. Do not normalize
all candidate labels together. The real checkpoint maps index 2 to entailment;
[verify the config](https://huggingface.co/joeddav/xlm-roberta-large-xnli/blob/main/config.json)
before changing models.

In [ ]:
# ==============================================================================
# SETUP / FIXTURES (Numerical exercises, not model outputs)
# ==============================================================================
label_logits = {"positive": [0.0, 0.0, math.log(6)],
                "negative": [math.log(6), 0.0, 0.0],
                "neutral": [0.0, 0.0, 0.0]}

# ==============================================================================
# YOUR IMPLEMENTATION
# ==============================================================================
def stable_softmax(logits):
    # TODO: Turn finite logits into a list of normalized probabilities.
    # Example inputs: logits=[0.0, 0.0, 0.0]
    # Example: -> [1/3, 1/3, 1/3]
    # Hint: Subtract the largest logit before exponentiating; divide by the total.
    # Hint: Check empty input before looking for a maximum.
    ### YOUR CODE HERE ###
    pass
    ### END CODE ###

def rank_entailment(label_logits, entailment_index=2):
    # TODO: Score each label independently using its selected NLI probability.
    # Example inputs: label_logits={"topic": [0.0, 0.0, 0.0]}, entailment_index=2
    # Example: -> {"topic": 1/3}
    # Hint: Reuse stable_softmax per label, select the index, then sort descending.
    # Hint: Keep stable ordering for tied scores; preserve empty input as empty.
    ### YOUR CODE HERE ###
    pass
    ### END CODE ###

# ==============================================================================
# TESTS & VERIFICATION
# ==============================================================================
run_check("uniform logits", lambda: stable_softmax([0.0, 0.0, 0.0]), [1/3, 1/3, 1/3])
run_check("large logits stay finite", lambda: stable_softmax([1000.0, 1000.0]), [0.5, 0.5])
run_check("unequal logits", lambda: stable_softmax([0.0, math.log(3)]), [0.25, 0.75])
run_check("empty logits", lambda: stable_softmax([]), [])
run_check("independent entailment scores", lambda: rank_entailment(label_logits),
          {"positive": 0.75, "neutral": 1/3, "negative": 0.125})
run_check("ranking order", lambda: list(rank_entailment(label_logits)), ["positive", "neutral", "negative"])
run_check("alternate entailment index", lambda: rank_entailment({"x": [math.log(3), 0.0]}, 0), {"x": 0.75})
run_check("stable ties", lambda: list(rank_entailment({"b": [0, 0, 0], "a": [0, 0, 0]})), ["b", "a"])
run_check("no candidates", lambda: rank_entailment({}), {})

### Connect Step 1 to the real model

- `classify(text, candidate_labels, hypothesis_template="This text is about {}.")`
  — **exact docs signature**. Use global `tok` and `model`; format each label into
  a hypothesis, tokenize the text/hypothesis pair with `return_tensors="pt"`
  and `truncation=True`, then call the model under `torch.no_grad()`. Apply
  `torch.softmax` over the last logit dimension, extract index 2 and sort the
  Python scores descending. It implements the same formula you just built.

Run `model.eval()` for reproducible inference. Keep this NLI model separate from
the embedding model in Step 2. These checks use the **three original sentences**.
They verify the output contract and compare to real forward-pass probabilities;
there are no invented expected labels or fixed model scores. The generic
"about" template is preserved from the docs; a sentiment-specific hypothesis
may work better. Inspect actual predictions before drawing conclusions.

In [ ]:
# ==============================================================================
# SETUP / FIXTURES (Real API; requires explicit opt-in)
# ==============================================================================
classification_texts = ["I love this product!", "मुझे यह उत्पाद पसंद है!", "J'adore ce produit !"]
candidate_labels = ["positive", "negative", "neutral"]
if RUN_XNLI:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        tok = AutoTokenizer.from_pretrained("joeddav/xlm-roberta-large-xnli")
        model = AutoModelForSequenceClassification.from_pretrained("joeddav/xlm-roberta-large-xnli")
        model.eval()
    except Exception as exc:
        tok = model = None
        print(f"❗ XNLI setup: {type(exc).__name__}: {exc} (Expected: real tokenizer and model)")

# ==============================================================================
# YOUR IMPLEMENTATION
# ==============================================================================
def classify(text, candidate_labels, hypothesis_template="This text is about {}."):
    # TODO: Rank candidate_labels by real NLI entailment for the input text.
    # Example inputs: text="I love this product!", candidate_labels=["positive", "negative", "neutral"]
    # Example: -> dict with those three labels and descending float scores in [0, 1].
    # Hint: Build a premise/hypothesis pair per label; use tok, model, no_grad and softmax.
    # Hint: The first batch item contains three NLI logits. Entailment is index 2.
    # Hint: Convert each selected tensor scalar to a Python float; keep empty candidates empty.
    ### YOUR CODE HERE ###
    pass
    ### END CODE ###

# ==============================================================================
# TESTS & VERIFICATION
# ==============================================================================
if RUN_XNLI and model is not None:
    run_check("checkpoint entailment index", lambda: model.config.label2id["entailment"], 2)
    run_check("no candidates", lambda: classify(classification_texts[0], []), {})
    for text in classification_texts:
        try:
            prediction = classify(text, candidate_labels)
            run_check(f"{text}: label set", lambda: set(prediction), set(candidate_labels))
            run_check("descending scores", lambda: list(prediction.values()), sorted(prediction.values(), reverse=True))
            run_check("finite probability scores", lambda: all(isinstance(v, float) and math.isfinite(v) and 0 <= v <= 1 for v in prediction.values()), True)
            # Independent numerical oracle from genuine logits, never mocked data.
            for label in candidate_labels:
                encoded = tok(text, f"This text is about {label}.", return_tensors="pt", truncation=True)
                with torch.no_grad():
                    expected = torch.softmax(model(**encoded).logits[0], dim=-1)[2].item()
                run_check(f"{text}: {label}", lambda label=label: prediction[label], expected, tol=1e-5)
        except Exception as exc:
            print(f"❗ classify: {type(exc).__name__}: {exc} (Expected: sorted label-to-score dict)")
else:
    print("❗ Real XNLI checks pending (Expected: consent flag, dependencies, model, and implementation).")

## Step 2: multilingual embedding space

Two vectors can point in the same direction despite different magnitudes.
Implement **practice helper** `cosine_similarity(a, b)` from scalar operations:

$$\cos(a,b)=\frac{\sum_{i=1}^d a_i b_i}{\sqrt{\sum_{i=1}^d a_i^2}\sqrt{\sum_{i=1}^d b_i^2}}.$$

Use equally sized, nonempty vectors. Raise `ValueError` for dimension mismatch,
empty vectors or a zero norm: cosine is undefined there. Do not silently truncate
unequal vectors. If both vectors are normalized, cosine reduces to their dot
product, explaining the docs' `np.dot` call.

In [ ]:
# ==============================================================================
# YOUR IMPLEMENTATION
# ==============================================================================
def cosine_similarity(a, b):
    # TODO: Return the cosine of the angle between finite vectors a and b.
    # Example inputs: a=[3.0, 4.0], b=[6.0, 8.0]
    # Example: -> 1.0
    # Hint: Check dimensions, accumulate dot product and squared norms, then divide.
    # Hint: Empty vectors, unequal dimensions and zero norms must raise ValueError.
    ### YOUR CODE HERE ###
    pass
    ### END CODE ###

# ==============================================================================
# TESTS & VERIFICATION
# ==============================================================================
run_check("same direction", lambda: cosine_similarity([3, 4], [6, 8]), 1.0)
run_check("orthogonal", lambda: cosine_similarity([1, 0], [0, 1]), 0.0)
run_check("opposite direction", lambda: cosine_similarity([1, 2, 2], [-1, -2, -2]), -1.0)
run_check("partial alignment", lambda: cosine_similarity([1, 0], [1, 1]), 1 / math.sqrt(2))
for a, b in [([], []), ([0, 0], [1, 0]), ([1, 2], [1])]:
    try:
        actual = cosine_similarity(a, b)
        check_test(f"invalid vectors {a}, {b}", actual, "ValueError")
    except ValueError:
        check_test(f"invalid vectors {a}, {b}", "ValueError", "ValueError")
    except Exception as exc:
        check_test("invalid-vector exception type", type(exc).__name__, "ValueError")

### Connect Step 2 to real multilingual embeddings

- `embedding_similarity(eng, other)` — **practice wrapper for the docs' inline
  loop**. Use global `embedding_model`, call `encode` on each one-sentence list
  with `normalize_embeddings=True`, select the first vector and return a Python
  float from `np.dot`. The formula is the unit-vector case of cosine above.

The exact four pairs below come from the docs. Compare your scalar cosine with
the real normalized-vector dot product. Translation similarity is an observation
to investigate, not a guaranteed threshold or a fabricated expected score.
See the [SentenceTransformer API](https://sbert.net/docs/package_reference/sentence_transformer/model.html).

In [ ]:
# ==============================================================================
# SETUP / FIXTURES (Exact docs pairs; genuine model only)
# ==============================================================================
pairs = [
    ("The cat is sleeping.", "Le chat dort."),
    ("The cat is sleeping.", "El gato está durmiendo."),
    ("The cat is sleeping.", "Die Katze schläft."),
    ("The cat is sleeping.", "The dog is barking."),
]
if RUN_EMBEDDINGS:
    try:
        from sentence_transformers import SentenceTransformer
        import numpy as np
        embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cpu")
    except Exception as exc:
        embedding_model = None
        print(f"❗ Embedding setup: {type(exc).__name__}: {exc} (Expected: real SentenceTransformer)")

# ==============================================================================
# YOUR IMPLEMENTATION
# ==============================================================================
def embedding_similarity(eng, other):
    # TODO: Encode both strings with the real embedding_model and compare normalized vectors.
    # Example inputs: eng="The cat is sleeping.", other="Le chat dort."
    # Example: -> a finite float in [-1, 1]; the actual value depends on model output.
    # Hint: Encode one-item lists with normalization, extract vectors, take their dot product.
    ### YOUR CODE HERE ###
    pass
    ### END CODE ###

# ==============================================================================
# TESTS & VERIFICATION
# ==============================================================================
if RUN_EMBEDDINGS and embedding_model is not None:
    for eng, other in pairs:
        try:
            left = embedding_model.encode([eng], normalize_embeddings=True)[0]
            right = embedding_model.encode([other], normalize_embeddings=True)[0]
            expected = float(np.dot(left, right))
            run_check("unit embedding norm", lambda: float(np.linalg.norm(left)), 1.0)
            run_check(f"scalar cosine: {other}", lambda: cosine_similarity(left.tolist(), right.tolist()), expected)
            run_check(f"real embedding similarity: {other}", lambda: embedding_similarity(eng, other), expected)
        except Exception as exc:
            print(f"❗ Embedding check: {type(exc).__name__}: {exc} (Expected: real normalized vectors)")
else:
    print("❗ Real embedding checks pending (Expected: consent flag, dependencies, model, and implementation).")

## Step 3: few-shot fine-tuning strategy

Before delegating training to `Trainer`, build two **practice helpers** to
understand what the training batch and classification loss mean:

- `pad_batch(examples, pad_token_id=0)` — each example has an `input_ids` list
  and scalar `label`. Return `input_ids`, `attention_mask`, and `labels` as
  Python lists. Pad on the right to $L=\max_i\lvert x_i\rvert$; mask
  $M_{ij}=1$ when $j<\lvert x_i\rvert$, otherwise $0$. Empty batch gives three
  empty lists. Construct new rows without mutating the originals.
- `cross_entropy(probabilities, label)` — $\mathcal L=-\ln(p_y)$, where `label`
  is the correct class index. Assume a valid distribution with $p_y>0$.

These labels are **one class ID per sentence**, not seq2seq target tokens.
Build the mask from original lengths: a token ID equal to the padding ID can
still be a real token in this deliberately general exercise. The later real
training exercise uses the tokenizer's padding ID and a real data collator.

In [ ]:
# ==============================================================================
# SETUP / FIXTURES (Token IDs for batch arithmetic only)
# ==============================================================================
batch_examples = [{"input_ids": [12, 45, 9], "label": 1},
                  {"input_ids": [23, 8], "label": 0}]
batch_snapshot = [{"input_ids": row["input_ids"][:], "label": row["label"]} for row in batch_examples]

# ==============================================================================
# YOUR IMPLEMENTATION
# ==============================================================================
def pad_batch(examples, pad_token_id=0):
    # TODO: Turn variable-length examples into equally wide rows, masks, and class labels.
    # Example inputs: examples=[{"input_ids": [7, 8], "label": 1}, {"input_ids": [4], "label": 0}]
    # Example: -> {"input_ids": [[7, 8], [4, 0]], "attention_mask": [[1, 1], [1, 0]], "labels": [1, 0]}
    # Hint: Find the batch maximum length, preserve original lengths, append padding to copies.
    # Hint: Handle an empty batch before taking a maximum; labels remain scalars per row.
    ### YOUR CODE HERE ###
    pass
    ### END CODE ###

def cross_entropy(probabilities, label):
    # TODO: Return the negative natural logarithm of the true class probability.
    # Example inputs: probabilities=[0.25, 0.75], label=1
    # Example: -> approximately 0.2876820725
    # Hint: Select the true class, then apply negative log; do not average over classes.
    ### YOUR CODE HERE ###
    pass
    ### END CODE ###

# ==============================================================================
# TESTS & VERIFICATION
# ==============================================================================
run_check("pad and mask", lambda: pad_batch(batch_examples),
          {"input_ids": [[12, 45, 9], [23, 8, 0]], "attention_mask": [[1, 1, 1], [1, 1, 0]], "labels": [1, 0]})
run_check("source examples unchanged", lambda: batch_examples, batch_snapshot)
run_check("nonzero padding ID", lambda: pad_batch(batch_examples, 77),
          {"input_ids": [[12, 45, 9], [23, 8, 77]], "attention_mask": [[1, 1, 1], [1, 1, 0]], "labels": [1, 0]})
run_check("empty batch", lambda: pad_batch([]), {"input_ids": [], "attention_mask": [], "labels": []})
run_check("empty row and real zero token", lambda: pad_batch([{"input_ids": [], "label": 0}, {"input_ids": [0], "label": 1}]),
          {"input_ids": [[0], [0]], "attention_mask": [[0], [1]], "labels": [0, 1]})
run_check("confident correct class", lambda: cross_entropy([0.25, 0.75], 1), 0.2876820725)
run_check("confident wrong class", lambda: cross_entropy([0.25, 0.75], 0), 1.3862943611)
run_check("certain correct class", lambda: cross_entropy([1.0, 0.0], 0), 0.0)

### Connect Step 3 to a real Trainer

- `few_shot_finetune(base_model, base_tokenizer, examples)` — **exact docs signature**.
  Create a real `Dataset.from_list`, map the nested tokenizer function, construct
  `TrainingArguments`, create `Trainer`, call `train()` and return `base_model`.
- Nested `tokenize_fn(ex)` — **exact docs signature**. Tokenize `ex["text"]`
  using `truncation=True`, `max_length=128`; attach `ex["label"]` as `labels`.

Preserve the docs' settings: `output_dir="out"`, batch size **8**, **5** epochs,
learning rate **2e-5**, `save_strategy="no"`. Disable external reporting with
`report_to="none"`. The docs omit batch padding: pass a real
`DataCollatorWithPadding(base_tokenizer)` to `Trainer` to handle unequal lengths.
Remove original dataset columns after mapping, retaining model input columns and
`labels`. These small robustness additions do not change the training recipe.
See the official [Trainer API](https://huggingface.co/docs/transformers/en/main_classes/trainer).

**Prepare the task model deliberately.** `task_model` must already have a head
whose class IDs match your examples. The Step 1 NLI head means
contradiction/neutral/entailment, not negative/positive sentiment; do not silently
reuse it as a sentiment head. Provide a separately initialized task classifier
and matching tokenizer, plus your learner-owned labeled target-language data.
The source docs supply no Step 3 dataset: no fake dataset or success score is
invented here. Tiny data can verify plumbing but cannot establish transfer quality.

The call updates the supplied model in place. Use a disposable training instance.
Set `RUN_FINETUNING=True` only after selecting data and reviewing resource costs.
Afterward evaluate on held-out examples **per language**; training loss alone does
not demonstrate that cross-lingual alignment was preserved.

In [ ]:
# ==============================================================================
# SETUP / FIXTURES (Supply real, task-appropriate objects yourself)
# ==============================================================================
# Assign these three variables before rerunning this cell; existing assignments survive.
task_model = globals().get("task_model")
task_tokenizer = globals().get("task_tokenizer")
few_shot_examples = globals().get("few_shot_examples", [])
training_dependencies_ready = False
if RUN_FINETUNING:
    try:
        from datasets import Dataset
        from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
        training_dependencies_ready = True
    except Exception as exc:
        print(f"❗ Training imports: {type(exc).__name__}: {exc} (Expected: real training libraries)")

# ==============================================================================
# YOUR IMPLEMENTATION
# ==============================================================================
def few_shot_finetune(base_model, base_tokenizer, examples):
    # TODO: Fine-tune the supplied classifier on examples using the docs' settings.
    # Example inputs: examples=[{"text": "J'adore ce produit !", "label": 1}, ...]
    # Example: -> the same base_model object with updated parameters after training.
    # Hint: Build a Dataset, map tokenize_fn, remove source columns, configure padding.
    # Hint: Batch 8, epochs 5, learning rate 2e-5, output out, no saves or external reporting.
    # Hint: Use the real Trainer, run training, then return the supplied model.
    def tokenize_fn(ex):
        # TODO: Encode ex["text"] and add its scalar class label to the tokenizer mapping.
        # Example inputs: ex={"text": "J'adore ce produit !", "label": 1}
        # Example: -> real tokenizer fields plus "labels": 1, at most 128 input tokens.
        # Hint: Enable truncation at 128 tokens; defer padding to the batch collator.
        ### YOUR CODE HERE ###
        pass
        ### END CODE ###

    ### YOUR CODE HERE ###
    pass
    ### END CODE ###

# ==============================================================================
# TESTS & VERIFICATION (Real execution only; no fake Trainer)
# ==============================================================================
if RUN_FINETUNING and training_dependencies_ready and task_model is not None and task_tokenizer is not None and few_shot_examples:
    try:
        import torch
        # Check class IDs and a genuine variable-length batch before expensive training.
        assert all(isinstance(ex["label"], int) and 0 <= ex["label"] < task_model.config.num_labels for ex in few_shot_examples), "Labels must match the task head"
        assert any(p.requires_grad for p in task_model.parameters()), "At least one parameter must be trainable"
        encoded_examples = []
        for ex in few_shot_examples[:8]:
            row = task_tokenizer(ex["text"], truncation=True, max_length=128)
            row["labels"] = ex["label"]
            encoded_examples.append(row)
        real_batch = DataCollatorWithPadding(task_tokenizer)(encoded_examples)
        check_test("real batch row count", real_batch["input_ids"].shape[0], len(encoded_examples))
        check_test("real class label shape", list(real_batch["labels"].shape), [len(encoded_examples)])
        # Small snapshots avoid a second full model copy in RAM.
        before = {name: p.detach().flatten()[:64].cpu().clone() for name, p in task_model.named_parameters() if p.requires_grad}
        trained_model = few_shot_finetune(task_model, task_tokenizer, few_shot_examples)
        check_test("returned original task model", trained_model is task_model, True)
        changed = any(not torch.equal(before[name], p.detach().flatten()[:64].cpu()) for name, p in task_model.named_parameters() if name in before)
        check_test("at least one sampled parameter changed", changed, True)
        print("Training smoke check only. Measure held-out accuracy separately for each language.")
    except Exception as exc:
        print(f"❗ Fine-tuning: {type(exc).__name__}: {exc} (Expected: trained real task model)")
else:
    print("❗ Fine-tuning checks pending (Expected: explicit opt-in, libraries, task model/tokenizer, labeled examples, and implementation).")

## Extra 1: typological feature similarity

The reference program explores source-language choice, a different question from
the three model exercises. Its table deliberately simplifies linguistics:
word order, script and family are teaching categories, not a complete description
of a language. No model is trained here.

- `similarity(a, b)` — **exact reference signature**. Look up both language keys
  in `LANGUAGE_FEATURES`; with $F$ the feature keys of the first language,
  $$s(a,b)=\frac{1}{|F|}\sum_{f\in F}\mathbf{1}[a_f=b_f].$$
  All fixture rows have the same three keys. Unknown languages raise `KeyError`,
  matching the reference. Do not silently invent features for missing languages.

In [ ]:
# ==============================================================================
# SETUP / FIXTURES (Exact LANGUAGE_FEATURES from code/main.py)
# ==============================================================================
LANGUAGE_FEATURES = {
    "english":  {"word_order": "SVO", "script": "Latin",   "family": "Germanic"},
    "german":   {"word_order": "SVO", "script": "Latin",   "family": "Germanic"},
    "french":   {"word_order": "SVO", "script": "Latin",   "family": "Romance"},
    "spanish":  {"word_order": "SVO", "script": "Latin",   "family": "Romance"},
    "italian":  {"word_order": "SVO", "script": "Latin",   "family": "Romance"},
    "hindi":    {"word_order": "SOV", "script": "Devanagari", "family": "Indic"},
    "marathi":  {"word_order": "SOV", "script": "Devanagari", "family": "Indic"},
    "bengali":  {"word_order": "SOV", "script": "Bengali",    "family": "Indic"},
    "urdu":     {"word_order": "SOV", "script": "Arabic",     "family": "Indic"},
    "arabic":   {"word_order": "VSO", "script": "Arabic",     "family": "Semitic"},
    "japanese": {"word_order": "SOV", "script": "Kanji",      "family": "Japonic"},
}

# ==============================================================================
# YOUR IMPLEMENTATION
# ==============================================================================
def similarity(a, b):
    # TODO: Return the fraction of matching features for language keys a and b.
    # Example inputs: a="marathi", b="hindi"
    # Example: -> 1.0; a="marathi", b="english" -> 0.0
    # Hint: Look up both rows, count equal values by feature key, divide by feature count.
    # Hint: Follow the reference's direct lookup behavior for unknown language keys.
    ### YOUR CODE HERE ###
    pass
    ### END CODE ###

# ==============================================================================
# TESTS & VERIFICATION
# ==============================================================================
run_check("Marathi / Hindi", lambda: similarity("marathi", "hindi"), 1.0)
run_check("Marathi / English", lambda: similarity("marathi", "english"), 0.0)
run_check("Marathi / Urdu", lambda: similarity("marathi", "urdu"), 2/3)
run_check("Arabic / Urdu", lambda: similarity("arabic", "urdu"), 1/3)
run_check("English / French", lambda: similarity("english", "french"), 2/3)
run_check("self similarity", lambda: similarity("japanese", "japanese"), 1.0)
try:
    actual = similarity("unknown", "english")
    check_test("unknown language", actual, "KeyError")
except KeyError:
    check_test("unknown language", "KeyError", "KeyError")
except Exception as exc:
    check_test("unknown-language exception type", type(exc).__name__, "KeyError")

## Extra 2: rank candidate source languages

- `rank_source_languages(target, candidates)` — **exact reference signature**.
  Construct $(c,s(\mathrm{target},c))$ for each $c\ne\mathrm{target}$, then sort
  descending by similarity. Return a list of tuples. Stable sorting preserves
  candidate order on ties; duplicate candidates remain duplicated, as in the
  reference. Empty candidates give `[]`.

This is a three-feature heuristic, not an implementation of LANGRANK or qWALS.
It generates hypotheses for experiments, not guaranteed transfer rankings.

In [ ]:
# ==============================================================================
# YOUR IMPLEMENTATION
# ==============================================================================
def rank_source_languages(target, candidates):
    # TODO: Return descending (source, similarity) tuples, excluding target itself.
    # Example inputs: target="marathi", candidates=["english", "hindi", "marathi"]
    # Example: -> [("hindi", 1.0), ("english", 0.0)]
    # Hint: Filter self, reuse similarity, then sort by score with stable ties.
    ### YOUR CODE HERE ###
    pass
    ### END CODE ###

# ==============================================================================
# TESTS & VERIFICATION
# ==============================================================================
run_check("exclude self and rank", lambda: rank_source_languages("marathi", ["english", "hindi", "marathi"]), [("hindi", 1.0), ("english", 0.0)])
run_check("tied sources keep input order", lambda: rank_source_languages("marathi", ["urdu", "bengali"]), [("urdu", 2/3), ("bengali", 2/3)])
run_check("no candidates", lambda: rank_source_languages("marathi", []), [])
run_check("only target", lambda: rank_source_languages("marathi", ["marathi"]), [])
run_check("duplicate sources", lambda: rank_source_languages("marathi", ["hindi", "hindi"]), [("hindi", 1.0), ("hindi", 1.0)])
run_check("reference top four", lambda: rank_source_languages("marathi", list(LANGUAGE_FEATURES))[:4],
          [("hindi", 1.0), ("bengali", 2/3), ("urdu", 2/3), ("japanese", 1/3)])

## Extra 3: simulated transfer accuracy

- `simulate_transfer_accuracy(target, source)` — **exact reference signature**:
  $$\widehat{A}(t,s)=\min(0.95,\;0.45+0.45\,s(t,s)).$$

This is a hand-written simulation, **not measured accuracy**, a fitted prediction,
or an API replacement. With the current similarity range $[0,1]$, the maximum
output is $0.90$, so the $0.95$ cap is never reached. Keep it to match the source.

In [ ]:
# ==============================================================================
# YOUR IMPLEMENTATION
# ==============================================================================
def simulate_transfer_accuracy(target, source):
    # TODO: Map target/source similarity to the reference's simulated accuracy.
    # Example inputs: target="marathi", source="hindi"
    # Example: -> 0.90; source="english" -> 0.45
    # Hint: Add a similarity-scaled boost of 0.45 to baseline 0.45, then cap at 0.95.
    ### YOUR CODE HERE ###
    pass
    ### END CODE ###

# ==============================================================================
# TESTS & VERIFICATION
# ==============================================================================
run_check("full match simulation", lambda: simulate_transfer_accuracy("marathi", "hindi"), 0.90)
run_check("zero match simulation", lambda: simulate_transfer_accuracy("marathi", "english"), 0.45)
run_check("two matches simulation", lambda: simulate_transfer_accuracy("urdu", "hindi"), 0.75)
run_check("one match simulation", lambda: simulate_transfer_accuracy("arabic", "urdu"), 0.60)
run_check("self simulation is still not certainty", lambda: simulate_transfer_accuracy("japanese", "japanese"), 0.90)

## Extra 4: assemble the reference report

- `main()` — **exact reference signature**. Use all feature-table keys as source
  candidates. For targets `marathi`, `urdu`, `arabic`, and `japanese`, print the
  top four source candidates, similarity with two decimals, and simulated
  accuracy as a percentage without decimals. Return `None`.

Reuse your functions. Print the reference header
`=== source language selection (qWALS-style similarity) ===`, then target lines
such as `target: marathi` and source lines such as
`source=hindi       sim=1.00  simulated_acc=90%`.
Finish with the two reference notes shown in the checks. This report must label
simulated scores honestly; a convincing report is not evidence of real accuracy.

In [ ]:
# ==============================================================================
# YOUR IMPLEMENTATION
# ==============================================================================
def main():
    # TODO: Compose the four-target source selection report using your Extra functions.
    # Example inputs: none; use global LANGUAGE_FEATURES.
    # Example: -> prints four target blocks, sixteen source rows, then two notes; returns None.
    # Hint: Rank all candidates per target, keep four, format similarity and simulated accuracy.
    # Hint: Use a source field width of 10. End with the reference's caveat and Marathi insight.
    ### YOUR CODE HERE ###
    pass
    ### END CODE ###

# ==============================================================================
# TESTS & VERIFICATION
# ==============================================================================
try:
    report_buffer = StringIO()
    with redirect_stdout(report_buffer):
        report_result = main()
    report = report_buffer.getvalue()
    print(report or "❗ main produced no report (Expected: four target blocks and caveat).")
    check_test("main return value", report_result, None)
    report_lines = [line.strip() for line in report.splitlines() if line.strip()]
    check_test("report header", report_lines[0] if report_lines else None, "=== source language selection (qWALS-style similarity) ===")
    check_test("target order", [line for line in report_lines if line.startswith("target:")],
               ["target: marathi", "target: urdu", "target: arabic", "target: japanese"])
    check_test("four sources per target overall", sum(line.startswith("source=") for line in report_lines), 16)
    check_test("Marathi top source", "source=hindi       sim=1.00  simulated_acc=90%" in report, True)
    check_test("simulation caveat", "note: real similarity comes from qWALS / lang2vec, not a 3-feature toy." in report, True)
    check_test("reference insight", "key insight: for Marathi, Hindi is a better source than English." in report, True)
except Exception as exc:
    print(f"❗ main: {type(exc).__name__}: {exc} (Expected: four-target simulation report)")

### Before you finish

1. Explain why three NLI probabilities sum to one, while three candidate-label
   entailment scores need not.
2. Explain why normalized embeddings allow a dot product instead of full cosine.
3. Show why padding changes tensor shape but must not add real content to attention.
4. Explain why a sentiment label `1` cannot automatically mean NLI label `1`.
5. Distinguish a toy transfer estimate, a passed API check, and measured held-out
   per-language accuracy.

Restart the kernel and run all cells to check reproducibility. With opt-in flags
off, completed offline exercises should pass and real-model checks stay explicitly
pending. They are not counted as verified model inference or training.